# VQA Datathon — Training, Verification & Submission Pipeline

This notebook:
1. Mounts Google Drive and installs exact competition-pinned dependencies.
2. Unpacks the dataset to Colab local scratch disk (fast I/O).
3. Automatically syncs latest model code and improvements from GitHub.
4. Trains the model on T4 GPU.
5. Runs the official Codabench ingestion program on val (2 CPU threads) to calculate FLOPs, parameters, accuracy, and composite score.
6. Builds the final `submission.zip` directly on your Google Drive.

## 1. Setup & Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/vqa_datathon'
PKG_ZIP = '/content/drive/MyDrive/student_package.zip'  # Adjust path if located elsewhere
WORK = '/content/vqa_datathon'

In [ ]:
%%capture pipout
!pip install torch==2.14.1 torchvision==0.29.1 transformers==5.20.2 pillow==12.3.0 numpy==2.2.6 tqdm==4.70.1 pyyaml==6.0.3

In [ ]:
import sys
PINS = {'torch': '2.14.1', 'torchvision': '0.29.1', 'transformers': '5.20.2', 'PIL': '12.3.0', 'numpy': '2.2.6'}
stale = [m for m, v in PINS.items()
         if m in sys.modules and getattr(sys.modules[m], '__version__', v).split('+')[0] != v]
if stale:
    print(f'Pinned versions installed, but old {stale} already loaded: restarting session.')
    import os, time; time.sleep(1); os.kill(os.getpid(), 9)
print('Library versions OK')

In [ ]:
import os, subprocess, torch, torchvision, transformers
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| CUDA', torch.cuda.is_available())
assert torch.__version__.startswith('2.14.1'), 'Restart runtime (Runtime -> Restart session) and rerun'

if not os.path.exists(WORK):
    if PKG_ZIP and os.path.exists(PKG_ZIP):
        subprocess.run(['unzip', '-q', PKG_ZIP, '-d', '/content'], check=True)
        os.rename('/content/student_package', WORK)
    else:
        subprocess.run(['cp', '-r', DRIVE, WORK], check=True)

os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
os.makedirs(f'{WORK}/data', exist_ok=True)
ck = f'{WORK}/data/checkpoints'
if not os.path.islink(ck):
    subprocess.run(['rm', '-rf', ck]); os.symlink(f'{DRIVE}/checkpoints', ck)

os.chdir(WORK)
print('Dataset ready in:', os.getcwd())

## 2. Sync Latest Code from GitHub
This pulls the latest model architectures, training scripts, and updates directly from the GitHub repository.

In [ ]:
!rm -rf /tmp/cv_repo
!git clone --depth 1 https://github.com/vedantcoder/cv-datathon.git /tmp/cv_repo
!cp -r /tmp/cv_repo/baseline /content/vqa_datathon/
!cp -r /tmp/cv_repo/eval /content/vqa_datathon/
!cp /tmp/cv_repo/config.yaml /content/vqa_datathon/
!rm -rf /tmp/cv_repo
print('Successfully synced latest model code and scripts from GitHub!')

## 3. Train Model (T4 GPU)
MobileNetV3-Small features (~135 MFLOPs, ~4.3M params) give an estimated **+13.7 point efficiency bonus** over the baseline.

In [ ]:
!python baseline/train_mbv3_small.py --epochs 45 --batch-size 128 --lr 2e-3

## 4. Build and Check Submission (2 CPU Threads)
Runs Codabench's exact ingestion program on 2,000 validation questions to measure params, FLOPs, accuracy, and composite score.

In [ ]:
MODEL_DIR = 'baseline/mbv3_small'
CKPT = 'data/checkpoints/mbv3_small.pt'
OUT_ZIP = f'{DRIVE}/submission_mbv3_small.zip'

!python eval/check_submission.py --model-dir {MODEL_DIR} --ckpt {CKPT} --out {OUT_ZIP} --n-val 2000 --threads 2

## 5. Download & Submit
Your verified `submission_mbv3_small.zip` is now in your Google Drive (`vqa_datathon/submission_mbv3_small.zip`).
Upload it to Codabench under **My Submissions**.